# Train + evaluate the full cascade (Colab or Kaggle GPU)

Fetch a SID_Set subset -> train Tiers 1/2/3 -> fit the fusion meta-model -> robustness eval -> download `outputs/`.

**Before running:** set a GPU runtime (Colab: Runtime -> Change runtime type -> T4 GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2, Internet On). Then Run All.

**Kaggle:** upload `TikTokSofa_project.zip` as a Dataset input first (+ Add Input -> Upload).

Fast-pass defaults below take ~15-25 min. For the final submission run, raise `TRAIN_SHARDS`, drop `MAX_PER_SHARD`/`MAX_IMG_DIM` to `None`, raise `EPOCHS` (see the config cell).

In [ ]:
# --- load the project (Colab + Kaggle; handles zip OR auto-extracted folder) ---
import os, glob, zipfile, shutil

DEST = ('/kaggle/working/TikTokSofa' if os.path.isdir('/kaggle/working')
        else '/content/TikTokSofa' if os.path.isdir('/content') else 'TikTokSofa')

def _locate():
    # A) already-extracted project tree (Kaggle unzips dataset archives on upload)
    for m in (glob.glob('/kaggle/input/**/src/train_fusion.py', recursive=True) +
              glob.glob('/content/**/src/train_fusion.py', recursive=True)):
        return 'dir', os.path.dirname(os.path.dirname(m))
    # B) a zip we extract ourselves
    for z in (glob.glob('/kaggle/input/**/*.zip', recursive=True) +
              glob.glob('/content/**/*.zip', recursive=True) + glob.glob('*.zip')):
        if 'tiktoksofa' in z.lower():
            return 'zip', z
    return None, None

if not os.path.exists(os.path.join(DEST, 'src/train_fusion.py')):
    kind, path = _locate()
    if kind == 'dir':
        shutil.copytree(path, DEST, dirs_exist_ok=True)
    elif kind == 'zip':
        with zipfile.ZipFile(path) as f: f.extractall(DEST)
    else:
        try:
            from google.colab import files
            up = files.upload()
            with zipfile.ZipFile(next(k for k in up if k.endswith('.zip'))) as f: f.extractall(DEST)
        except Exception:
            raise SystemExit('Add the "tiktoksofa project" dataset as a Kaggle input first (right sidebar -> + Add Input -> Your Work).')

os.chdir(DEST)
print('cwd:', os.getcwd())
assert os.path.exists('src/train_fusion.py') and os.path.exists('scripts/prep_sid_set.py'), 'project files not found'

In [ ]:
# --- dependencies (Colab already has torch + CUDA + pandas; install the rest) ---
!pip -q install timm open_clip_torch 'huggingface_hub>=0.23' pyarrow scikit-learn opencv-python-headless python-dotenv
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# --- config ---
# FAST FIRST PASS defaults. Once you've seen the whole notebook run green, scale up for the
# real submission: TRAIN_SHARDS 20-40, MAX_PER_SHARD None, MAX_IMG_DIM None, EPOCHS 5-8.
TRAIN_SHARDS  = 4
VAL_SHARDS    = 1
MAX_PER_SHARD = 250     # hard cap on images extracted per shard (None = whole shard)
MAX_IMG_DIM   = 384     # downscale longer side before Tier 2 forensic features -> big speedup
                        # (None = native res; the block-grid / double-JPEG signal is stronger at
                        #  native res, so use None for the final run)
EPOCHS        = 4       # Tier 1 fine-tuning epochs
N_AUGMENTS_T2 = 1       # augmented variants per image for Tier 2 (2 for the final run)
EXCLUDE_TAMPERED = False   # True = drop SID_Set label==2 (locally edited) instead of calling them fake

import os
os.environ.setdefault('HF_TOKEN', '')   # optional: paste a free HF token for faster downloads

In [ ]:
# --- fetch a SID_Set subset -> data/raw/sid_set/{train,val}/{real,fake} ---
tamper_flag = '--exclude-tampered' if EXCLUDE_TAMPERED else ''
cap_flag    = f'--max-per-shard {MAX_PER_SHARD}' if MAX_PER_SHARD else ''
!python scripts/prep_sid_set.py --out data/raw/sid_set --train-shards $TRAIN_SHARDS --val-shards $VAL_SHARDS $cap_flag $tamper_flag
!echo; echo 'train:'; ls data/raw/sid_set/train/real | wc -l; ls data/raw/sid_set/train/fake | wc -l
!echo 'val:';   ls data/raw/sid_set/val/real   | wc -l; ls data/raw/sid_set/val/fake   | wc -l

In [ ]:
R_TRAIN, F_TRAIN = 'data/raw/sid_set/train/real', 'data/raw/sid_set/train/fake'
R_VAL,   F_VAL   = 'data/raw/sid_set/val/real',   'data/raw/sid_set/val/fake'
DIM_FLAG = f'--max-image-dim {MAX_IMG_DIM}' if MAX_IMG_DIM else ''   # passed to every stage that touches Tier 2 features

## Tier 2 — hand-built forensic features (CPU, fast)

In [ ]:
!python -m src.frequency.train_svm --real-dir $R_TRAIN --fake-dir $F_TRAIN --classifier rf \
    --n-augments-per-image $N_AUGMENTS_T2 $DIM_FLAG --out outputs/tier2_classifier.joblib

## Tier 3 — frozen CLIP ViT-B/32 + linear probe

In [ ]:
!python -m src.semantic.train_probe --real-dir $R_TRAIN --fake-dir $F_TRAIN --out outputs/tier3_clip_probe.joblib

## Tier 1 — EfficientNet-B0 fine-tune (GPU)

In [ ]:
!python -m src.model.train --real-dir $R_TRAIN --fake-dir $F_TRAIN --epochs $EPOCHS --batch-size 32 --device cuda --out outputs/tier1_efficientnet_b0.pt

## Fusion meta-model — fit on the HELD-OUT val split

In [ ]:
!python -m src.train_fusion --real-dir $R_VAL --fake-dir $F_VAL $DIM_FLAG --scores-cache outputs/_fusion_scores.npz

## Evaluation

In [ ]:
# Robustness table (isolated + compound transforms, per tier).
# FIRST PASS: SID_Set held-out split (below).
# FINAL SUBMISSION: per spec section 2 + repo README, run against the designated benchmark set
#   instead -- data/raw/val_coco/val2017 + data/raw/val_dalle (benchmark-only, never trained on).
!python -m src.eval.robustness --real-dir $R_VAL --fake-dir $F_VAL --n-samples 120 $DIM_FLAG --out outputs/robustness_summary.csv
import pandas as pd; pd.read_csv('outputs/robustness_summary.csv')

In [ ]:
# Cascade inference on the held-out fake images -> predictions.json (schema demo + error analysis input).
!python -m src.infer $F_VAL --out outputs/predictions.json --always-escalate $DIM_FLAG
import json; print(json.dumps(json.load(open('outputs/predictions.json'))[:3], indent=2))

### Cross-generator holdout (the real generalization test)
SID_Set is not split by generator family, so this needs WildFake laid out as `<family>/{real,fake}/` (see `data/README.md`). Skip on the first pass; run it for the final submission and record the gap in `docs/shortcut_learning_check.md`.
```
!python -m src.eval.cross_generator --data-root data/raw/wildfake --holdout-family <family>
```

## Download the results

In [ ]:
import os, shutil
shutil.make_archive('/kaggle/working/trained_outputs' if os.path.isdir('/kaggle/working') else 'trained_outputs',
                    'zip', 'outputs')
try:
    from google.colab import files
    files.download('trained_outputs.zip')                      # Colab -> downloads to your computer
except Exception:
    if os.path.isdir('/kaggle/working'):
        print('Kaggle: right sidebar -> Output -> /kaggle/working/trained_outputs.zip -> download icon.')
        print('        Or Save Version (Quick Save), then the notebook Output tab.')
    else:
        print('Saved:', os.path.abspath('trained_outputs.zip'))

Unzip `trained_outputs.zip` into your local `outputs/`. The `*.pt` / `*.joblib` weights are gitignored (don't commit them); **do** commit `robustness_summary.csv` and `predictions.json`, and paste the AUC numbers into `docs/error_analysis.md`.